# NB11 — Robustness / Sensitivity

**NB11 does not redefine any NB09 primary result. It tests their stability.**

```
CANONICAL_BASE_SHA     925697ccc06514f86ae48952ab42f18bae3fd70f
NB09_RESULT_SHA        820943391fb8601f64e79d9ae1b13fc60eeaa694
NB09_AUDIT_SHA         27814139a43bfbe3d4eb2e646e97c0107acfa582  NB09_RESULT_AUDIT_PASS
AUDITED_PROTOCOL_SHA   81b682527c587f64c817b7dab74f538f16bf9152
PROTOCOL_AUDIT_SHA     33c59d58a00202e961aa68a2f9e23fcf27033004
```

Every family below comes from `KOEN-TP-RS-001` §24 or from the NB11 registry frozen in
`ssot_nb09/01_PRE_NB09_PROTOCOL_v001.md` §9 **before** NB09 ran. No family was invented in
response to a favourable NB09 result, and **no sensitivity is selected because its fit is larger.**

Every subset rule and every column variant in §1 is stated **before any sensitivity is fitted**.

## 1. Pre-registration — subsets, variants and estimators

### 1.1 Realized-support notes, measured before fitting

- `source_tier` has **one** realized level (`A`, 3,835,988). SSOT §24.3's "full accepted vs
  curated-source-only" therefore has **no realized contrast variable**. The closest realized
  analogue is the per-source split, which is run instead and labelled as such — not as a
  curated-tier contrast.
- `duplicate_group_id` is **all-singleton** (3,835,988 distinct of 3,835,988). It does not
  implement `LR-01`, so it cannot serve as a dependence grouping key (§1.4).
- `translation_quality_review_flag` and `named_entity_heavy_flag` are identically false;
  D-03 and D-05 `analysis_warning_flag` are identically false.

In [1]:
from __future__ import annotations
import json, hashlib, platform, csv, subprocess
from pathlib import Path
import numpy as np, duckdb, scipy, pyarrow, pyarrow.parquet as pq
from scipy import stats
from scipy.linalg import cho_factor, cho_solve
from tokenization_premium.telemetry import RuntimeTelemetry

ROOT = Path("/home/sieg/projects-wsl/KOEN_nb11_20260818")
REG, RUNTIME = ROOT / "data/registry", ROOT / ".runtime/nb11"
RUNTIME.mkdir(parents=True, exist_ok=True)
for d in ("outputs/reports", "outputs/tables", "outputs/manifests", "docs/results/nb11"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)

SHAS = {"CANONICAL_BASE_SHA": "925697ccc06514f86ae48952ab42f18bae3fd70f",
        "NB09_RESULT_SHA": "820943391fb8601f64e79d9ae1b13fc60eeaa694",
        "NB09_AUDIT_SHA": "27814139a43bfbe3d4eb2e646e97c0107acfa582",
        "AUDITED_PROTOCOL_SHA": "81b682527c587f64c817b7dab74f538f16bf9152",
        "PROTOCOL_AUDIT_SHA": "33c59d58a00202e961aa68a2f9e23fcf27033004"}
CONTRACT = json.loads((ROOT / "ssot_nb09/02_NB09_MODEL_MATRIX_CONTRACT_v001.json").read_text("utf-8"))
SEEDS    = json.loads((ROOT / "ssot_nb09/03_NB09_SEED_REGISTRY_v001.json").read_text("utf-8"))
NB09     = json.loads((ROOT / "outputs/reports/NB09_EXPLANATORY_RESULTS_v001.json").read_text("utf-8"))
EXPECTED_N, EXPECTED_PAIR_SET = CONTRACT["cohort"]["N"], CONTRACT["cohort"]["pair_set_hash"]
MODELS   = {m: v["continuous"] for m, v in CONTRACT["models"].items()}
OUTCOMES = {"A": CONTRACT["outcomes"]["A"]["column"], "B": CONTRACT["outcomes"]["B"]["column"]}
NB09_R2  = {(r["outcome"], r["model"]): r["r2"] for r in NB09["models"]}
NB09_BLK = {(b["outcome"], b["comparison"]): b for b in NB09["block_comparisons"]}
print("NB09 primary R2 carried in for comparison:", {k: round(v, 6) for k, v in NB09_R2.items()})

NB09 primary R2 carried in for comparison: {('A', 'M0'): 0.028625, ('A', 'M1'): 0.649578, ('A', 'M2'): 0.656169, ('A', 'M2A'): 0.655423, ('A', 'M3'): 0.806939, ('B', 'M0'): 0.182, ('B', 'M1'): 0.509203, ('B', 'M2'): 0.518433, ('B', 'M2A'): 0.517389, ('B', 'M3'): 0.7296}


### 1.2 Row subsets — rules fixed here

| id | rule | SSOT family |
|---|---|---|
| `FULL` | the frozen analysis cohort | baseline |
| `KNOWN_DIR` | `translation_direction != 'UNKNOWN'` | §24.6 |
| `NO_ANOMALY` | none of `unicode_anomaly_flag`, `high_digit_ratio_flag`, `high_punctuation_ratio_flag`, `lang_side_anomaly_review_flag` | §24.4 |
| `CENTRAL_LENGTH` | `length_stratum IN ('Q2','Q3','Q4')` | §24.5 |
| `SOURCE_025` | `source_id` starting `025` | §24.3 analogue |
| `SOURCE_026` | `source_id` starting `026` | §24.3 analogue |

`short_text_flag` / `long_text_flag` are **not** used in `NO_ANOMALY` because they are the length
strata that `CENTRAL_LENGTH` already handles; `script_mix_flag` is excluded because script mixing is
a substantive modelled feature, not an anomaly.

### 1.3 Column variants — rules fixed here

| id | change | registry |
|---|---|---|
| `V0` | NB09 primary, unchanged | baseline |
| `V1_SM01` | replace `{ko,en}_script_type_count` with `{ko,en}_script_multi = 1{type ≥ 2}`; keep `switch_count` | NB11 #10 |
| `V2_M3DENS` | replace `{ko,en}_chunk_count_log` with `{ko,en}_chunk_density_log = ln k − ln C` | NB11 #11 |
| `V3_M3DENS_LOGMCB` | `V2` **and** `mean_chunk_bytes` on the log scale | NB11 #11, rank-checked first |
| `V4_B_SSOT182` | Outcome **B only**: drop `log_code_point_ratio` and `log_byte_density_ratio` | §1A, SPEC-02 route |
| `V5_SRC_DOM` | replace `source_domain_cell` with additive `source_id + domain` | §24.9 / `CR` §4.3, rank-checked |
| `V6_SPEC01` | add the held-out D-02 block (URL/email/emoji/code-like flags, space-run counts, bytes-per-grapheme) | NB11 #13 |

### 1.4 Dependence

`duplicate_group_id` is all-singleton, so it does not implement `LR-01` and is **not** used as a
grouping key. No grouping is invented here.

```
NB11_DEPENDENCE_SENSITIVITY = DEPENDENCY_PENDING_PRE_NB10_GROUPING
```

In [2]:
EXPECTED_SHA = {
 "D-01": ("PAIR_REGISTRY_v002.parquet",       "95f523d11b0e8fcfd761dee949f082e9b4590b919801441fbcfa3426010bec52"),
 "D-02": ("REP_FEATURES_v002.parquet",        "dfae8e01cd3fe2ca949d8754678e508203ad1a7aa6abea418008a33ac650d309"),
 "D-03": ("MORPH_FEATURES_KIWI_v001.parquet", "0fe5bd74e3993a7141c5c33ea78e71b2c66e3ecd296544bde2615acb43e50f7d"),
 "D-04": ("TOKEN_O200K_BASE_v001.parquet",    "1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7"),
 "D-05": ("CHUNK_O200K_BASE_v001.parquet",    "bfa98bd6cf7ee8b7254c469aed3e259ce43cc8f0529153347ca4c2c3fc1944ab")}
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()
ARTIFACTS = {k: {"filename": fn, "sha256": sha256_file(REG / fn), "expected_sha256": exp}
             for k, (fn, exp) in EXPECTED_SHA.items()}
for k, a in ARTIFACTS.items():
    a["match"] = a["sha256"] == a["expected_sha256"]; print(f"{k} {'MATCH' if a['match'] else 'MISMATCH'}")
assert all(a["match"] for a in ARTIFACTS.values()), "ARTIFACT_IDENTITY_FAIL"
print("ARTIFACT_IDENTITY = 5/5")

D-01 MATCH
D-02 MATCH
D-03 MATCH
D-04 MATCH
D-05 MATCH
ARTIFACT_IDENTITY = 5/5


## 2. Materialize — NB09 columns plus the pre-registered variant columns

In [3]:
P = f"read_parquet('{(REG/'PAIR_REGISTRY_v002.parquet').as_posix()}')"
R = f"read_parquet('{(REG/'REP_FEATURES_v002.parquet').as_posix()}')"
M = f"read_parquet('{(REG/'MORPH_FEATURES_KIWI_v001.parquet').as_posix()}')"
T = f"read_parquet('{(REG/'TOKEN_O200K_BASE_v001.parquet').as_posix()}')"
K = f"read_parquet('{(REG/'CHUNK_O200K_BASE_v001.parquet').as_posix()}')"
NB09_COLS = ",\n  ".join(sorted(set(sum(MODELS.values(), []))) )
SELECT = f'''
SELECT t.pair_id, t.log_token_premium, t.log_compression_penalty,
  t.log_code_point_ratio, t.log_byte_density_ratio,
  0.5*(ln(r.ko_codepoint_count)+ln(r.en_codepoint_count)) AS pair_log_size,
  r.ko_whitespace_density - r.en_whitespace_density        AS delta_whitespace_density,
  r.ko_latin_share, r.ko_digit_share, r.ko_punctuation_share, r.ko_symbol_other_share,
  r.en_hangul_share, r.en_digit_share, r.en_punctuation_share, r.en_symbol_other_share,
  CAST(r.ko_script_type_count AS DOUBLE) AS ko_script_type_count,
  CAST(r.ko_script_switch_count AS DOUBLE) AS ko_script_switch_count,
  CAST(r.en_script_type_count AS DOUBLE) AS en_script_type_count,
  CAST(r.en_script_switch_count AS DOUBLE) AS en_script_switch_count,
  m.morpheme_density, m.particle_ratio, m.ending_ratio, m.deriv_affix_ratio,
  m.function_morpheme_ratio,
  ln(k.ko_chunk_count) AS ko_chunk_count_log, ln(k.en_chunk_count) AS en_chunk_count_log,
  k.ko_mean_chunk_bytes, k.ko_p50_chunk_bytes, k.ko_p90_chunk_bytes,
  CAST(k.ko_max_chunk_bytes AS DOUBLE) AS ko_max_chunk_bytes,
  k.en_mean_chunk_bytes, k.en_p50_chunk_bytes, k.en_p90_chunk_bytes,
  CAST(k.en_max_chunk_bytes AS DOUBLE) AS en_max_chunk_bytes,
  CAST(k.ko_max_tokens_per_chunk AS DOUBLE) AS ko_max_tokens_per_chunk,
  CAST(k.en_max_tokens_per_chunk AS DOUBLE) AS en_max_tokens_per_chunk,
  k.ko_chunk_type_share_number, k.ko_chunk_type_share_punctuation, k.ko_chunk_type_share_whitespace,
  k.en_chunk_type_share_number, k.en_chunk_type_share_punctuation, k.en_chunk_type_share_whitespace,
  -- pre-registered variant columns
  CASE WHEN r.ko_script_type_count >= 2 THEN 1.0 ELSE 0.0 END AS ko_script_multi,
  CASE WHEN r.en_script_type_count >= 2 THEN 1.0 ELSE 0.0 END AS en_script_multi,
  ln(k.ko_chunk_count) - ln(r.ko_codepoint_count) AS ko_chunk_density_log,
  ln(k.en_chunk_count) - ln(r.en_codepoint_count) AS en_chunk_density_log,
  ln(k.ko_mean_chunk_bytes) AS ko_mean_chunk_bytes_log,
  ln(k.en_mean_chunk_bytes) AS en_mean_chunk_bytes_log,
  CAST(r.ko_url_flag AS DOUBLE) AS ko_url_flag, CAST(r.en_url_flag AS DOUBLE) AS en_url_flag,
  CAST(r.ko_email_flag AS DOUBLE) AS ko_email_flag, CAST(r.en_email_flag AS DOUBLE) AS en_email_flag,
  CAST(r.ko_emoji_flag AS DOUBLE) AS ko_emoji_flag, CAST(r.en_emoji_flag AS DOUBLE) AS en_emoji_flag,
  CAST(r.ko_code_like_flag AS DOUBLE) AS ko_code_like_flag,
  CAST(r.en_code_like_flag AS DOUBLE) AS en_code_like_flag,
  CAST(r.ko_space_run_count AS DOUBLE) AS ko_space_run_count,
  CAST(r.en_space_run_count AS DOUBLE) AS en_space_run_count,
  r.ko_bytes_per_grapheme, r.en_bytes_per_grapheme,
  -- subset keys
  p.source_id || '-' || p.domain AS source_domain_cell, p.translation_direction,
  p.source_id, p.domain, p.length_stratum,
  (p.unicode_anomaly_flag OR p.high_digit_ratio_flag OR p.high_punctuation_ratio_flag
   OR p.lang_side_anomaly_review_flag) AS anomaly_any
FROM {T} t JOIN {P} p ON p.pair_id=t.pair_id JOIN {R} r ON r.pair_id=t.pair_id
           JOIN {M} m ON m.pair_id=t.pair_id JOIN {K} k ON k.pair_id=t.pair_id
ORDER BY t.pair_id
'''
MATRIX = (RUNTIME / "nb11_matrix.parquet").as_posix()
con = duckdb.connect(); con.execute("PRAGMA memory_limit='3GB'"); con.execute("PRAGMA threads=4")
con.execute(f"PRAGMA temp_directory='{(RUNTIME/'spill').as_posix()}'")
with RuntimeTelemetry(run_id="NB11_MATERIALIZE", stage="MATERIALIZE", total=EXPECTED_N) as tel:
    con.execute(f"COPY ({SELECT}) TO '{MATRIX}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    A = f"read_parquet('{MATRIX}')"
    n_rows, n_dist = con.execute(f"SELECT count(*), count(DISTINCT pair_id) FROM {A}").fetchone()
    pair_set = con.execute(f"SELECT md5(string_agg(pair_id,'' ORDER BY pair_id)) FROM {A}").fetchone()[0]
    ordered = con.execute(f"SELECT count(*) FROM (SELECT pair_id, lag(pair_id) OVER () prev FROM {A}) "
                          "WHERE prev IS NOT NULL AND pair_id < prev").fetchone()[0]
    tel.update(n_rows)
TEL_MAT = {k: v for k, v in tel.summary().items() if k != "samples"}
assert (n_rows, n_dist, pair_set, ordered) == (EXPECTED_N, EXPECTED_N, EXPECTED_PAIR_SET, 0)
print(f"N={n_rows} pair-set={pair_set} ORDER_BY_PAIR_ID_ASSERTED=True")

# SSOT §24.1/§24.2 — text-variant families: bound them without regenerating any artifact
TEXTBOUND = con.execute(f'''SELECT
  sum(CASE WHEN ko_text_raw <> ko_text_analysis THEN 1 ELSE 0 END),
  sum(CASE WHEN en_text_raw <> en_text_analysis THEN 1 ELSE 0 END),
  sum(CASE WHEN ko_text_nfc <> ko_text_analysis THEN 1 ELSE 0 END),
  sum(CASE WHEN en_text_nfc <> en_text_analysis THEN 1 ELSE 0 END), count(*)
  FROM {P} p JOIN {T} t ON t.pair_id=p.pair_id''').fetchone()
print("text-variant differing-row counts (raw KO/EN, nfc KO/EN, N):", TEXTBOUND)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

N=3835988 pair-set=d9660d654ee449e4d0c23a0070225274 ORDER_BY_PAIR_ID_ASSERTED=True


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

text-variant differing-row counts (raw KO/EN, nfc KO/EN, N): (1112, 154, 0, 0, 3835988)


## 3. One streaming pass — sufficient statistics for every subset

Every OLS sensitivity in §1.3 is a submatrix solve on an extended Gram, so all six subsets and all
column variants are covered by a single pass. `RQ1`'s median is taken per subset by an exact
DuckDB quantile, not by a model.

In [4]:
BASE_CONT = list(dict.fromkeys(sum(MODELS.values(), [])))
EXTRA = ["ko_script_multi", "en_script_multi", "ko_chunk_density_log", "en_chunk_density_log",
         "ko_mean_chunk_bytes_log", "en_mean_chunk_bytes_log",
         "ko_url_flag", "en_url_flag", "ko_email_flag", "en_email_flag",
         "ko_emoji_flag", "en_emoji_flag", "ko_code_like_flag", "en_code_like_flag",
         "ko_space_run_count", "en_space_run_count", "ko_bytes_per_grapheme", "en_bytes_per_grapheme"]
ALL_CONT = BASE_CONT + EXTRA
LEV = CONTRACT["reference_levels"]
cell_counts = dict(con.execute(f"SELECT source_domain_cell, count(*) FROM {A} GROUP BY 1").fetchall())
dir_counts  = dict(con.execute(f"SELECT translation_direction, count(*) FROM {A} GROUP BY 1").fetchall())
dom_counts  = dict(con.execute(f"SELECT domain, count(*) FROM {A} GROUP BY 1").fetchall())
src_counts  = dict(con.execute(f"SELECT source_id, count(*) FROM {A} GROUP BY 1").fetchall())
CELL_REF, DIR_REF = LEV["source_domain_cell"], LEV["translation_direction"]
CELL_LV = [l for l in sorted(cell_counts, key=lambda x: (-cell_counts[x], x)) if l != CELL_REF]
DIR_LV  = [l for l in sorted(dir_counts,  key=lambda x: (-dir_counts[x],  x)) if l != DIR_REF]
DOM_REF = max(dom_counts, key=dom_counts.get); SRC_REF = max(src_counts, key=src_counts.get)
DOM_LV  = [l for l in sorted(dom_counts, key=lambda x: (-dom_counts[x], x)) if l != DOM_REF]
SRC_LV  = [l for l in sorted(src_counts, key=lambda x: (-src_counts[x], x)) if l != SRC_REF]
DUM = ([f"cell::{l}" for l in CELL_LV] + [f"dir::{l}" for l in DIR_LV]
       + [f"dom::{l}" for l in DOM_LV] + [f"src::{l}" for l in SRC_LV])
DESIGN = ["__intercept__"] + ALL_CONT + DUM
IDX = {c: i for i, c in enumerate(DESIGN)}
print("extended design width:", len(DESIGN), "| dummies:", len(DUM))

SUBSETS = ["FULL", "KNOWN_DIR", "NO_ANOMALY", "CENTRAL_LENGTH", "SOURCE_025", "SOURCE_026"]
READ = ALL_CONT + list(OUTCOMES.values()) + ["source_domain_cell", "translation_direction",
                                             "source_id", "domain", "length_stratum", "anomaly_any"]
PF = pq.ParquetFile(MATRIX)

def batches(bs):
    for b in PF.iter_batches(batch_size=bs, columns=READ):
        d = b.to_pydict(); m = len(d[ALL_CONT[0]])
        X = np.empty((m, len(DESIGN))); X[:, 0] = 1.0
        for c in ALL_CONT: X[:, IDX[c]] = np.asarray(d[c], dtype=np.float64)
        cell = np.asarray(d["source_domain_cell"], dtype=object); dr = np.asarray(d["translation_direction"], dtype=object)
        dom = np.asarray(d["domain"], dtype=object); src = np.asarray(d["source_id"], dtype=object)
        for l in CELL_LV: X[:, IDX[f"cell::{l}"]] = (cell == l)
        for l in DIR_LV:  X[:, IDX[f"dir::{l}"]]  = (dr == l)
        for l in DOM_LV:  X[:, IDX[f"dom::{l}"]]  = (dom == l)
        for l in SRC_LV:  X[:, IDX[f"src::{l}"]]  = (src == l)
        Y = {k: np.asarray(d[v], dtype=np.float64) for k, v in OUTCOMES.items()}
        ls = np.asarray(d["length_stratum"], dtype=object); an = np.asarray(d["anomaly_any"], dtype=bool)
        masks = {"FULL": np.ones(m, bool), "KNOWN_DIR": dr != "UNKNOWN", "NO_ANOMALY": ~an,
                 "CENTRAL_LENGTH": np.isin(ls, ["Q2", "Q3", "Q4"]),
                 "SOURCE_025": np.array([s.startswith("025") for s in src]),
                 "SOURCE_026": np.array([s.startswith("026") for s in src])}
        yield X, Y, masks

pd_ = len(DESIGN)
ST = {s: {"n": 0, "G": np.zeros((pd_, pd_)), "Xty": {k: np.zeros(pd_) for k in OUTCOMES},
          "yty": {k: 0.0 for k in OUTCOMES}, "ysum": {k: 0.0 for k in OUTCOMES}} for s in SUBSETS}
with RuntimeTelemetry(run_id="NB11_PASS_SUFFSTAT", stage="SUFFICIENT_STATISTICS", total=EXPECTED_N) as tel:
    for X, Y, masks in batches(200_000):
        for s in SUBSETS:
            mk = masks[s]
            if not mk.any(): continue
            Xs = X[mk]; ST[s]["G"] += Xs.T @ Xs; ST[s]["n"] += int(mk.sum())
            for k in OUTCOMES:
                ys = Y[k][mk]
                ST[s]["Xty"][k] += Xs.T @ ys; ST[s]["yty"][k] += float(ys @ ys); ST[s]["ysum"][k] += float(ys.sum())
        tel.update(X.shape[0])
TEL_SS = {k: v for k, v in tel.summary().items() if k != "samples"}
print({s: ST[s]["n"] for s in SUBSETS})

extended design width: 68 | dummies: 10


{'FULL': 3835988, 'KNOWN_DIR': 3785441, 'NO_ANOMALY': 3728608, 'CENTRAL_LENGTH': 2387120, 'SOURCE_025': 2485963, 'SOURCE_026': 1350025}


## 4. Variant column sets, rank-checked before any fit

In [5]:
# 교체는 원래 열이 실제로 있는 모형에만 적용한다. M0/M1처럼 대상 열이 없는 모형에 대체 열을
# 얹으면 그 모형 자체가 달라져, RQ3/RQ4 비교가 변이가 아니라 다른 사다리가 되어 버린다.
def swap(cols, drop, add):
    if not any(c in drop for c in cols):
        return list(cols)
    out = [c for c in cols if c not in drop]
    return out + [a for a in add if a not in out]

SPEC01 = ["ko_url_flag", "en_url_flag", "ko_email_flag", "en_email_flag", "ko_emoji_flag",
          "en_emoji_flag", "ko_code_like_flag", "en_code_like_flag", "ko_space_run_count",
          "en_space_run_count", "ko_bytes_per_grapheme", "en_bytes_per_grapheme"]
VARIANTS = {}
VARIANTS["V0"] = {m: (MODELS[m], "cell") for m in MODELS}
VARIANTS["V1_SM01"] = {m: (swap(MODELS[m], {"ko_script_type_count", "en_script_type_count"},
                                ["ko_script_multi", "en_script_multi"]), "cell") for m in MODELS}
VARIANTS["V2_M3DENS"] = {m: (swap(MODELS[m], {"ko_chunk_count_log", "en_chunk_count_log"},
                                  ["ko_chunk_density_log", "en_chunk_density_log"]), "cell") for m in MODELS}
VARIANTS["V3_M3DENS_LOGMCB"] = {m: (swap(VARIANTS["V2_M3DENS"][m][0],
                                         {"ko_mean_chunk_bytes", "en_mean_chunk_bytes"},
                                         ["ko_mean_chunk_bytes_log", "en_mean_chunk_bytes_log"]), "cell")
                                for m in MODELS}
VARIANTS["V4_B_SSOT182"] = {m: ([c for c in MODELS[m]
                                 if c not in {"log_code_point_ratio", "log_byte_density_ratio"}], "cell")
                            for m in MODELS}
VARIANTS["V5_SRC_DOM"] = {m: (MODELS[m], "srcdom") for m in MODELS}
# SPEC-01은 표면/표현 블록이므로 M1 이상에만 얹는다. M0에 얹으면 RQ3가 변이가 아니라
# 다른 baseline 비교가 된다.
VARIANTS["V6_SPEC01"] = {m: ((MODELS[m] + SPEC01) if m != "M0" else MODELS[m], "cell") for m in MODELS}

def colidx(cont, catmode):
    dm = ([f"cell::{l}" for l in CELL_LV] if catmode == "cell"
          else [f"dom::{l}" for l in DOM_LV] + [f"src::{l}" for l in SRC_LV])
    return [IDX["__intercept__"]] + [IDX[c] for c in cont] + [IDX[d] for d in dm] + [IDX[f"dir::{l}"] for l in DIR_LV]

ZV_TOL = 1e-12
# subset 안에서 관측 지지가 없는 열(빈 dummy)이나 상수 열은 변수가 아니라 빈 열이다.
# G5 protocol §8의 zero-variance 구조적 제거를 subset마다 같은 결정론적 규칙으로 적용한다.
# 계수·p-value·R2·성능은 이 판정에 일절 쓰지 않는다.
def prune(sub, cols):
    G = ST[sub]["G"]; n_ = max(ST[sub]["n"], 1)
    keep, dropped = [], []
    for j in cols:
        if j == IDX["__intercept__"]:
            keep.append(j); continue
        gjj = G[j, j]; mean = G[IDX["__intercept__"], j] / n_
        ss = gjj - n_ * mean * mean
        if ss / max(gjj, 1.0) <= ZV_TOL:
            dropped.append(DESIGN[j])
        else:
            keep.append(j)
    return keep, dropped

def rank_of(sub, cols):
    G = ST[sub]["G"][np.ix_(cols, cols)]
    sc = np.sqrt(np.maximum(np.diag(G), 0) / max(ST[sub]["n"], 1)); sc[sc == 0] = 1.0
    Gs = G / np.outer(sc, sc); ev = np.linalg.eigvalsh(Gs)
    tol = max(ST[sub]["n"], len(cols)) * np.finfo(float).eps * ev[-1]
    return int((ev > tol).sum()), float(np.sqrt(ev[-1] / max(ev[0], 1e-300))), Gs, sc

# 변이가 의도대로만 다른지 먼저 확인한다: 대상 열이 없는 모형은 V0와 열 집합이 같아야 한다
VARIANT_SHAPE = {}
for v, spec in VARIANTS.items():
    for m in MODELS:
        base, cur = set(MODELS[m]), set(spec[m][0])
        VARIANT_SHAPE[f"{v}/{m}"] = {"p_continuous_base": len(base), "p_continuous_variant": len(cur),
                                     "added": sorted(cur - base), "removed": sorted(base - cur),
                                     "identical_to_primary": base == cur}
for k, v in VARIANT_SHAPE.items():
    if not v["identical_to_primary"]:
        print(f"  variant column change {k}: -{v['removed']} +{v['added']}")

# 구조적 pruning을 모든 subset x 변이 x 모형에 동일하게 적용하고 전부 기록한다
PRUNE_LOG, RANKCHECK = {}, {}
for v, spec in VARIANTS.items():
    for sub in SUBSETS:
        for m in MODELS:
            raw = colidx(*spec[m]); kept, dropped = prune(sub, raw)
            r, cond, _, _ = rank_of(sub, kept)
            rec = {"variant": v, "subset_id": sub, "model": m, "original_p": len(raw),
                   "dropped_zero_variance_columns": dropped, "effective_p": len(kept),
                   "resulting_rank": r, "full_rank": r == len(kept),
                   "deficiency_after_structural_pruning": len(kept) - r,
                   "condition_number_standardized": cond}
            PRUNE_LOG[f"{v}/{sub}/{m}"] = rec
            if sub == "FULL": RANKCHECK[f"{v}/{m}"] = rec
ZV_REMOVED = sum(len(r["dropped_zero_variance_columns"]) for r in PRUNE_LOG.values())
NOT_IDENT = {k: r for k, r in PRUNE_LOG.items() if not r["full_rank"]}
# 왜 비식별인지 구분한다. 두 원인은 성격이 전혀 다르다.
for k, r in NOT_IDENT.items():
    causes = []
    if r["subset_id"] != "FULL" and cell_counts and any(
            ST[r["subset_id"]]["G"][IDX[f"cell::{l}"], IDX[f"cell::{l}"]] == 0 for l in CELL_LV) \
            and ST[r["subset_id"]]["n"] > 0 and r["variant"] != "V5_SRC_DOM":
        # 참조 cell 자체가 subset에 없으면 남은 cell dummy 합이 절편과 같아진다
        present = [l for l in CELL_LV if ST[r["subset_id"]]["G"][IDX[f"cell::{l}"], IDX[f"cell::{l}"]] > 0]
        tot = sum(ST[r["subset_id"]]["G"][IDX[f"cell::{l}"], IDX[f"cell::{l}"]] for l in present)
        if abs(tot - ST[r["subset_id"]]["n"]) < 0.5:
            causes.append("REFERENCE_CELL_ABSENT_IN_SUBSET_DUMMIES_SATURATE_INTERCEPT")
    if r["variant"] == "V3_M3DENS_LOGMCB" and r["model"] == "M3":
        causes.append("EXACT_IDENTITY_ln_k_PLUS_ln_mean_chunk_bytes_EQUALS_ln_bytes")
    r["non_identifiability_causes"] = causes or ["UNCLASSIFIED"]
BLOCKED = sorted({k.split("/")[0] for k in NOT_IDENT})
# subset 수준 실패는 primary 사양(V0)이 실제로 깨지는 subset만을 뜻한다.
SUBSET_FAIL_V0 = sorted({r["subset_id"] for r in NOT_IDENT.values() if r["variant"] == "V0"})
SUBSET_FAIL_ANY_VARIANT = sorted({r["subset_id"] for r in NOT_IDENT.values()})
print(f"zero-variance columns removed across all subset x variant x model designs: {ZV_REMOVED}")
for sub in SUBSETS:
    ex = PRUNE_LOG[f"V0/{sub}/M3"]
    print(f"  {sub:15s} V0/M3 p {ex['original_p']} -> {ex['effective_p']} "
          f"(dropped {len(ex['dropped_zero_variance_columns'])}) rank={ex['resulting_rank']}")
print("\nnot identifiable AFTER structural pruning (genuine, not emptiness):")
for k, r in NOT_IDENT.items():
    print(f"  {k}: effective_p={r['effective_p']} rank={r['resulting_rank']} "
          f"def={r['deficiency_after_structural_pruning']}")
print("variants affected:", BLOCKED)
print("subsets where the PRIMARY specification V0 is not identifiable:", SUBSET_FAIL_V0)
print("subsets touched by any variant's non-identifiability:", SUBSET_FAIL_ANY_VARIANT)

  variant column change V1_SM01/M1: -['en_script_type_count', 'ko_script_type_count'] +['en_script_multi', 'ko_script_multi']
  variant column change V1_SM01/M2: -['en_script_type_count', 'ko_script_type_count'] +['en_script_multi', 'ko_script_multi']
  variant column change V1_SM01/M2A: -['en_script_type_count', 'ko_script_type_count'] +['en_script_multi', 'ko_script_multi']
  variant column change V1_SM01/M3: -['en_script_type_count', 'ko_script_type_count'] +['en_script_multi', 'ko_script_multi']
  variant column change V2_M3DENS/M3: -['en_chunk_count_log', 'ko_chunk_count_log'] +['en_chunk_density_log', 'ko_chunk_density_log']
  variant column change V3_M3DENS_LOGMCB/M3: -['en_chunk_count_log', 'en_mean_chunk_bytes', 'ko_chunk_count_log', 'ko_mean_chunk_bytes'] +['en_chunk_density_log', 'en_mean_chunk_bytes_log', 'ko_chunk_density_log', 'ko_mean_chunk_bytes_log']
  variant column change V4_B_SSOT182/M1: -['log_byte_density_ratio', 'log_code_point_ratio'] +[]
  variant column change

## 5. Fit every admissible variant × subset (OLS)

In [6]:
def fit_sub(sub, cols, out):
    cols, _dropped = prune(sub, cols)          # 적합 전에 구조적으로 빈 열을 제거한다
    n_ = ST[sub]["n"]; p = len(cols)
    r, cond, Gs, sc = rank_of(sub, cols)
    if r != p: return None
    c = cho_factor(Gs, lower=True)
    b_s = cho_solve(c, ST[sub]["Xty"][out][cols] / sc)
    ssr = float(ST[sub]["yty"][out] - 2 * b_s @ (ST[sub]["Xty"][out][cols] / sc) + b_s @ (Gs @ b_s))
    sst = float(ST[sub]["yty"][out] - n_ * (ST[sub]["ysum"][out] / n_) ** 2)
    r2 = 1.0 - ssr / sst
    return {"n": n_, "p": p, "rank": r, "condition_number_standardized": cond,
            "ssr": ssr, "sst": sst, "r2": r2,
            "adj_r2": 1.0 - (1.0 - r2) * (n_ - 1) / (n_ - p),
            "beta": b_s / sc, "cols": cols, "chol": c, "scale": sc}

CMP = [("RQ3", "M1", "M0"), ("RQ4", "M2", "M1"), ("RQ5", "M3", "M2"), ("SENS_M2A", "M2A", "M1")]
ROWS, FITSTORE = [], {}
for v, spec in VARIANTS.items():
    outs = ["B"] if v == "V4_B_SSOT182" else list(OUTCOMES)
    for sub in SUBSETS:
        for o in outs:
            fits = {m: fit_sub(sub, colidx(*spec[m]), o) for m in MODELS}
            FITSTORE[(v, sub, o)] = fits
            for rq, full, red in CMP:
                ff, fr = fits[full], fits[red]
                if ff is None or fr is None:
                    pl = PRUNE_LOG[f"{v}/{sub}/{full}"]
                    ROWS.append({"variant": v, "subset": sub, "outcome": o, "comparison": rq,
                        "status": "NOT_IDENTIFIABLE_EXACT_IDENTITY", "n": ST[sub]["n"],
                        "effective_p_full": pl["effective_p"], "rank_full": pl["resulting_rank"],
                        "dropped_zero_variance_columns_full": len(pl["dropped_zero_variance_columns"])})
                    continue
                d = ff["r2"] - fr["r2"]
                base = NB09_BLK.get((o, rq), {})
                ROWS.append({"variant": v, "subset": sub, "outcome": o, "comparison": rq,
                    "status": "OK", "n": ff["n"], "p_full": ff["p"], "p_reduced": fr["p"],
                    "r2_full": ff["r2"], "r2_reduced": fr["r2"], "delta_r2": d,
                    "partial_r2": d / (1.0 - fr["r2"]),
                    "condition_number_full": ff["condition_number_standardized"],
                    "dropped_zero_variance_columns_full":
                        len(PRUNE_LOG[f"{v}/{sub}/{full}"]["dropped_zero_variance_columns"]),
                    "nb09_primary_delta_r2": base.get("delta_r2"),
                    "nb09_primary_partial_r2": base.get("partial_r2"),
                    "delta_r2_ratio_vs_primary": (d / base["delta_r2"]) if base.get("delta_r2") else None})
SPAN_CHECK = {}
for o in OUTCOMES:
    a = FITSTORE[("V0", "FULL", o)]["M3"]; b = FITSTORE[("V2_M3DENS", "FULL", o)]["M3"]
    if a and b:
        SPAN_CHECK[o] = {"r2_V0": a["r2"], "r2_V2": b["r2"], "abs_diff": abs(a["r2"] - b["r2"]),
            "condition_V0": a["condition_number_standardized"], "condition_V2": b["condition_number_standardized"],
            "predicted": ("ln k = chunk_density_log + ln C, and ln C_KO / ln C_EN are already spanned by "
                          "pair_log_size and log_code_point_ratio, so V2 is a change of basis within the "
                          "same column space: R-squared must be identical and only conditioning changes.")}
        print(f"  span check M3/{o}: |R2(V0)-R2(V2)| = {SPAN_CHECK[o]['abs_diff']:.3e}  "
              f"cond {a['condition_number_standardized']:.2f} -> {b['condition_number_standardized']:.2f}")
print(f"{len(ROWS)} sensitivity rows; blocked = {sum(1 for r in ROWS if r['status'] != 'OK')}")
for r in ROWS:
    if r["variant"] in ("V1_SM01", "V2_M3DENS", "V4_B_SSOT182") and r["subset"] == "FULL" and r["status"] == "OK":
        print(f"  {r['variant']:16s} {r['outcome']} {r['comparison']:9s} dR2={r['delta_r2']:.6f} "
              f"partial={r['partial_r2']:.6f} (primary dR2={r['nb09_primary_delta_r2']})")

  span check M3/A: |R2(V0)-R2(V2)| = 1.193e-13  cond 1172.81 -> 653.36
  span check M3/B: |R2(V0)-R2(V2)| = 2.652e-13  cond 1172.81 -> 653.36
312 sensitivity rows; blocked = 54
  V1_SM01          A RQ3       dR2=0.620997 partial=0.639297 (primary dR2=0.6209535712798299)
  V1_SM01          A RQ4       dR2=0.006573 partial=0.018761 (primary dR2=0.0065902675051974224)
  V1_SM01          A RQ5       dR2=0.150748 partial=0.438470 (primary dR2=0.15076997613708454)
  V1_SM01          A SENS_M2A  dR2=0.005828 partial=0.016632 (primary dR2=0.005844583542528148)
  V1_SM01          B RQ3       dR2=0.327264 partial=0.400078 (primary dR2=0.327203008675225)
  V1_SM01          B RQ4       dR2=0.009207 partial=0.018761 (primary dR2=0.009230258752895404)
  V1_SM01          B RQ5       dR2=0.211136 partial=0.438470 (primary dR2=0.2111668290900761)
  V1_SM01          B SENS_M2A  dR2=0.008162 partial=0.016632 (primary dR2=0.008185861705469422)
  V2_M3DENS        A RQ3       dR2=0.620954 partial=0.639252 (

## 6. Huber — SSOT §24.8 robust-estimator family

Streaming IRLS with the standard tuning constant `1.345·σ̂`, `σ̂` from the MAD of the OLS residuals.
Quantile regression is **not executed in this pass** and the reason is recorded rather than the
family being quietly dropped.

In [7]:
HUBER_MODELS, HUBER_ITERS, HUBER_K = ["M0", "M1", "M2", "M3"], 8, 1.345
hb = {(m, o): FITSTORE[("V0", "FULL", o)][m]["beta"].copy() for m in HUBER_MODELS for o in OUTCOMES}
hcols = {m: colidx(*VARIANTS["V0"][m]) for m in HUBER_MODELS}
sigma = {}
with RuntimeTelemetry(run_id="NB11_HUBER_IRLS", stage="MAD", total=EXPECTED_N) as tel:
    absres = {k: [] for k in hb}
    for X, Y, _ in batches(200_000):
        for (m, o) in hb: absres[(m, o)].append(np.abs(Y[o] - X[:, hcols[m]] @ hb[(m, o)]))
        tel.update(X.shape[0])
    for k in hb: sigma[k] = float(np.median(np.concatenate(absres[k])) / 0.6744897501960817)
    del absres
    for it in range(HUBER_ITERS):
        tel.set_stage(f"IRLS_{it + 1}")
        acc = {k: [np.zeros((len(hcols[k[0]]),) * 2), np.zeros(len(hcols[k[0]]))] for k in hb}
        for X, Y, _ in batches(200_000):
            for (m, o) in hb:
                Xc = X[:, hcols[m]]; e = Y[o] - Xc @ hb[(m, o)]
                d = HUBER_K * sigma[(m, o)]
                w = np.minimum(1.0, d / np.maximum(np.abs(e), 1e-300))
                Xw = Xc * w[:, None]
                acc[(m, o)][0] += Xc.T @ Xw; acc[(m, o)][1] += Xw.T @ Y[o]
        for k in hb:
            Gk, bk = acc[k]
            sc = np.sqrt(np.maximum(np.diag(Gk), 0) / ST["FULL"]["n"]); sc[sc == 0] = 1.0
            hb[k] = cho_solve(cho_factor(Gk / np.outer(sc, sc), lower=True), bk / sc) / sc
TEL_HUB = {k: v for k, v in tel.summary().items() if k != "samples"}

HUBER = {}
for (m, o), b in hb.items():
    ols = FITSTORE[("V0", "FULL", o)][m]["beta"]
    common = [DESIGN[c] for c in hcols[m]]
    rel = float(np.max(np.abs(b - ols) / np.maximum(np.abs(ols), 1e-12)))
    HUBER[f"{m}/{o}"] = {"model": m, "outcome": o, "estimator": "Huber IRLS",
        "tuning_constant": HUBER_K, "iterations": HUBER_ITERS, "sigma_mad": sigma[(m, o)],
        "max_relative_coef_shift_vs_ols": rel,
        "sign_agreement_with_ols": float(np.mean(np.sign(b) == np.sign(ols))),
        "n_terms": len(common)}
    print(f"  Huber {m:3s}/{o} sigma={sigma[(m,o)]:.6f} max|rel shift|={rel:.4f} "
          f"sign agreement={HUBER[f'{m}/{o}']['sign_agreement_with_ols']:.4f}")
QUANTILE_STATUS = {"family": "SSOT §24.8 quantile regression", "status": "NOT_EXECUTED_THIS_PASS",
  "reason": ("No streaming L1/quantile solver is available in this environment that converges "
             "reliably at N = 3,835,988 with p up to 45 within the single-heavy-job budget. "
             "Subsampling would introduce a sample-selection seed that was not pre-registered. "
             "Recorded as an open NB11 family rather than dropped."),
  "not_a_substitute": "Huber addresses heavy-tail sensitivity; it does not answer a quantile question."}
print("\nquantile:", QUANTILE_STATUS["status"])

  Huber M0 /A sigma=0.202262 max|rel shift|=0.6648 sign agreement=1.0000
  Huber M0 /B sigma=0.156158 max|rel shift|=0.1024 sign agreement=1.0000
  Huber M1 /A sigma=0.117546 max|rel shift|=3.7450 sign agreement=0.9565
  Huber M1 /B sigma=0.117546 max|rel shift|=3.7450 sign agreement=0.9565
  Huber M2 /A sigma=0.116530 max|rel shift|=3.1806 sign agreement=0.9259
  Huber M2 /B sigma=0.116530 max|rel shift|=3.1806 sign agreement=0.9259
  Huber M3 /A sigma=0.093137 max|rel shift|=6.2697 sign agreement=0.9111
  Huber M3 /B sigma=0.093137 max|rel shift|=6.2697 sign agreement=0.9111

quantile: NOT_EXECUTED_THIS_PASS


## 7. RQ1 direction stability across the pre-registered subsets

In [8]:
RQ1 = {}
SUBSQL = {"FULL": "TRUE", "KNOWN_DIR": "translation_direction <> 'UNKNOWN'",
          "NO_ANOMALY": "NOT anomaly_any", "CENTRAL_LENGTH": "length_stratum IN ('Q2','Q3','Q4')",
          "SOURCE_025": "source_id LIKE '025%'", "SOURCE_026": "source_id LIKE '026%'"}
for s, w in SUBSQL.items():
    n_, med, pos, tie, neg = con.execute(
        f"SELECT count(*), median(log_token_premium), "
        f"sum(CASE WHEN log_token_premium>0 THEN 1 ELSE 0 END), "
        f"sum(CASE WHEN log_token_premium=0 THEN 1 ELSE 0 END), "
        f"sum(CASE WHEN log_token_premium<0 THEN 1 ELSE 0 END) FROM {A} WHERE {w}").fetchone()
    RQ1[s] = {"n": n_, "median_logTP": float(med), "median_TP": float(np.exp(med)),
              "share_TP_gt_1": pos / n_, "positive": pos, "tie": tie, "negative": neg,
              "median_positive": bool(med > 0)}
    print(f"  {s:15s} n={n_:>9,} median logTP={med:.10f} median TP={np.exp(med):.6f} P(TP>1)={pos/n_:.4f}")

  FULL            n=3,835,988 median logTP=0.2876820725 median TP=1.333333 P(TP>1)=0.8799


  KNOWN_DIR       n=3,785,441 median logTP=0.2876820725 median TP=1.333333 P(TP>1)=0.8798


  NO_ANOMALY      n=3,728,608 median logTP=0.2876820725 median TP=1.333333 P(TP>1)=0.8888


  CENTRAL_LENGTH  n=2,387,120 median logTP=0.2984929886 median TP=1.347826 P(TP>1)=0.8922
  SOURCE_025      n=2,485,963 median logTP=0.2744368457 median TP=1.315789 P(TP>1)=0.8336


  SOURCE_026      n=1,350,025 median logTP=0.3087354816 median TP=1.361702 P(TP>1)=0.9650


## 8. Claim register — direction, magnitude, and what is not testable

In [9]:
def rng_of(o, rq):
    vals = [r for r in ROWS if r["status"] == "OK" and r["outcome"] == o and r["comparison"] == rq
            and r["variant"] != "V4_B_SSOT182"]
    d = [r["delta_r2"] for r in vals]; pr = [r["partial_r2"] for r in vals]
    return {"n_specifications": len(vals), "delta_r2_min": min(d), "delta_r2_max": max(d),
            "partial_r2_min": min(pr), "partial_r2_max": max(pr),
            "all_positive": all(x > 0 for x in d)}

CLAIMS = {}
med_ok = all(v["median_positive"] for v in RQ1.values())
mr = [v["median_logTP"] for v in RQ1.values()]
CLAIMS["RQ1"] = {"claim": "Median(log_token_premium) > 0",
  "direction": "DIRECTION_STABLE" if med_ok else "SPECIFICATION_SENSITIVE",
  "magnitude": "MAGNITUDE_STABLE",
  "effect_range": {"median_logTP_min": min(mr), "median_logTP_max": max(mr),
                   "median_TP_min": float(np.exp(min(mr))), "median_TP_max": float(np.exp(max(mr))),
                   "share_TP_gt_1_min": min(v["share_TP_gt_1"] for v in RQ1.values()),
                   "share_TP_gt_1_max": max(v["share_TP_gt_1"] for v in RQ1.values())},
  "subsets_tested": list(RQ1), "note": "NB08 already closed RQ1; this is subset direction stability."}
for rq in ["RQ3", "RQ4", "RQ5"]:
    a, b = rng_of("A", rq), rng_of("B", rq)
    v4 = [r for r in ROWS if r["variant"] == "V4_B_SSOT182" and r["outcome"] == "B"
          and r["comparison"] == rq and r["subset"] == "FULL" and r["status"] == "OK"]
    prim = NB09_BLK[("A", rq)]["delta_r2"]
    lo, hi = a["delta_r2_min"] / prim, a["delta_r2_max"] / prim
    CLAIMS[rq] = {"claim": {"RQ3": "M1 - M0 representation/surface block",
                            "RQ4": "M2 - M1 morphology block",
                            "RQ5": "M3 - M2 regex-chunk mechanism block"}[rq],
      "direction": "DIRECTION_STABLE" if a["all_positive"] and b["all_positive"] else "SPECIFICATION_SENSITIVE",
      "magnitude": ("MAGNITUDE_STABLE" if 0.75 <= lo and hi <= 1.35 else "MAGNITUDE_ATTENUATED"),
      "outcome_A_range": a, "outcome_B_range": b,
      "ratio_vs_nb09_primary_A": {"min": lo, "max": hi},
      "outcome_B_ssot18_2_variant": ({"delta_r2": v4[0]["delta_r2"], "partial_r2": v4[0]["partial_r2"]}
                                     if v4 else "BLOCKED_OR_ABSENT")}
CLAIMS["SSOT_24_1_raw_text"] = {"status": "NOT_TESTABLE_AT_NB11",
  "reason": "requires regenerating D-02 and D-04 on ko/en_text_raw; no approved decision authorizes "
            "artifact regeneration, and NB11 may not create a new measurement artifact.",
  "bound": {"ko_rows_differing_from_analysis_text": TEXTBOUND[0],
            "en_rows_differing_from_analysis_text": TEXTBOUND[1], "N": TEXTBOUND[4]}}
CLAIMS["SSOT_24_2_nfc_only"] = {"status": "NOT_TESTABLE_AT_NB11", "reason": CLAIMS["SSOT_24_1_raw_text"]["reason"],
  "bound": {"ko_rows_differing_from_analysis_text": TEXTBOUND[2],
            "en_rows_differing_from_analysis_text": TEXTBOUND[3], "N": TEXTBOUND[4]}}
CLAIMS["SSOT_24_3_curated_source"] = {"status": "REALIZED_SUPPORT_ABSENT",
  "reason": "source_tier has a single realized level 'A' over the whole cohort, so no curated-vs-full "
            "contrast exists. The per-source subsets SOURCE_025 / SOURCE_026 are run as the closest "
            "realized analogue and are labelled as such, not as a curated-tier contrast."}
CLAIMS["SSOT_24_9_source_random"] = {"status": "FEASIBLE_SUPPORT_IS_FIXED_ONLY",
  "reason": "two realized source levels; the additive source_id + domain specification (V5_SRC_DOM) "
            "is run as the feasible fixed alternative to source_domain_cell.",
  "V5_rank": {k: v for k, v in RANKCHECK.items() if k.startswith("V5")}}
v4 = {rq: next((r for r in ROWS if r["variant"] == "V4_B_SSOT182" and r["subset"] == "FULL"
                and r["comparison"] == rq and r["status"] == "OK"), None) for rq in ["RQ4", "RQ5"]}
CLAIMS["V2_M3DENS"] = {"label": "SAME_SPAN_REPARAMETERIZATION",
  "r2_difference_M3": {o: SPAN_CHECK[o]["abs_diff"] for o in SPAN_CHECK},
  "condition_primary": SPAN_CHECK["A"]["condition_V0"], "condition_reparameterized": SPAN_CHECK["A"]["condition_V2"],
  "block_conclusion_changed": False,
  "reading": ("ln k = chunk_density_log + ln C and ln C_KO / ln C_EN are already spanned by "
              "pair_log_size and log_code_point_ratio, so this is a change of basis inside the same "
              "column space. It cannot change any block claim; it only improves conditioning.")}
CLAIMS["V3_M3DENS_LOGMCB"] = {"label": "NOT_IDENTIFIABLE_EXACT_IDENTITY",
  "rank": {k: {"effective_p": r["effective_p"], "rank": r["resulting_rank"]}
           for k, r in PRUNE_LOG.items() if k.startswith("V3_M3DENS_LOGMCB/FULL")},
  "reading": ("ln k + ln(mean_chunk_bytes) = ln B per side, and ln B_KO - ln B_EN is already spanned "
              "by log_code_point_ratio + log_byte_density_ratio, so logging both chunk terms creates "
              "an exact identity. The sensitivity is stopped, not repaired by deleting a substantive "
              "variable."),
  "repaired_by_deletion": False}
CLAIMS["V4_B_SSOT182"] = {"label": "SECONDARY_SENSITIVITY", "promoted_to_primary": False,
  "outcome": "B only; Outcome A untouched",
  "RQ4": {"primary_common_ladder": {"delta_r2": NB09_BLK[("B", "RQ4")]["delta_r2"],
                                    "partial_r2": NB09_BLK[("B", "RQ4")]["partial_r2"]},
          "ssot18_2_sensitivity": ({"delta_r2": v4["RQ4"]["delta_r2"],
                                    "partial_r2": v4["RQ4"]["partial_r2"]} if v4["RQ4"] else None)},
  "RQ5": {"primary_common_ladder": {"delta_r2": NB09_BLK[("B", "RQ5")]["delta_r2"],
                                    "partial_r2": NB09_BLK[("B", "RQ5")]["partial_r2"]},
          "ssot18_2_sensitivity": ({"delta_r2": v4["RQ5"]["delta_r2"],
                                    "partial_r2": v4["RQ5"]["partial_r2"]} if v4["RQ5"] else None)},
  "classification": ("OUTCOME_B_MORPHOLOGY = SPECIFICATION_SENSITIVE" if v4["RQ4"] and
                     v4["RQ4"]["partial_r2"] > 2.0 * NB09_BLK[("B", "RQ4")]["partial_r2"]
                     else "OUTCOME_B_MORPHOLOGY = SPECIFICATION_STABLE"),
  "reading": ("Under the approved common ladder Outcome B's fit is algebraically tied to Outcome A "
              "(B-N01). Removing the two representation ratios, as SSOT §18.2's illustrative form "
              "does, breaks that tie, so the morphology block's incremental value for compression "
              "asymmetry is reported on its own terms. This is a sensitivity, not a corrected "
              "primary model.")}
CLAIMS["dependence"] = {"status": "DEPENDENCY_PENDING_PRE_NB10_GROUPING",
  "evidence": "duplicate_group_id is all-singleton (3,835,988 distinct of 3,835,988) and therefore "
              "does not implement LR-01. No grouping key is invented here."}
for k in ["RQ1", "RQ3", "RQ4", "RQ5"]:
    c = CLAIMS[k]; print(f"  {k}: {c['direction']} / {c['magnitude']}")

  RQ1: DIRECTION_STABLE / MAGNITUDE_STABLE
  RQ3: DIRECTION_STABLE / MAGNITUDE_STABLE
  RQ4: DIRECTION_STABLE / MAGNITUDE_STABLE
  RQ5: DIRECTION_STABLE / MAGNITUDE_ATTENUATED


## 9. Persist

In [10]:
CODE_SHA = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
RESTRICTIONS = dict(NB09["claim_restrictions"])
RESTRICTIONS["NB11_ROLE"] = "stability testing only; NB09 primary results are not redefined"
RESTRICTIONS["SENSITIVITY_SELECTION"] = "pre-registered before fitting; never chosen for a larger fit"
BOOTSTRAP_STATUS = {"nb11_bootstrap_ci_engine": "NOT_USED",
  "reason": ("Claude-A has an open methodological review (A6) of the NB09 delta-R-squared bootstrap "
             "CI construction. NB11 therefore claims no bootstrap interval: stability is judged from "
             "point-estimate ranges across the pre-registered specifications in section 1, and from "
             "rank, conditioning and estimator-family comparisons."),
  "nb09_point_estimates_used": ("R-squared, delta R-squared and partial R-squared point estimates only; "
                                "these are closed and are not affected by A6"),
  "on_a6_closure": ("If A6 requires a corrected paired construction, adopt the independently validated "
                    "one for all later bootstrap block comparisons. No NB11 conclusion here depends on "
                    "the held construction, so none needs revisiting for that reason.")}
RESTRICTIONS["NB11_BOOTSTRAP_CI"] = "NOT_USED — pending Claude-A review A6"
SSOT29 = {"N_pairs": ST["FULL"]["n"], "source_count": len(src_counts), "domain_count": len(dom_counts),
  "estimator": "fixed-effects OLS; Huber IRLS for the SSOT §24.8 family",
  "se_type": "not recomputed at NB11 — NB11 reports effect-size ranges (see A-R01 note)",
  "random_fixed_effects": "fixed only; two realized source levels",
  "reference_levels": {"source_domain_cell": CELL_REF, "translation_direction": DIR_REF,
                       "domain": DOM_REF, "source_id": SRC_REF,
                       "ko_script_composition": "ko_hangul_share", "en_script_composition": "en_latin_share",
                       "ko_chunk_type": "ko_chunk_type_share_letter", "en_chunk_type": "en_chunk_type_share_letter"},
  "missingness_handling": "none present; fail-closed, no imputation, no row deletion",
  "preprocessing_version_id": {"protocol": "PRE_NB09_PROTOCOL_v001", "code_sha": CODE_SHA}}
SUMMARY = {"artifact_id": "NB11_ROBUSTNESS_SUMMARY_v001", **SHAS, "code_sha": CODE_SHA,
  "cohort": {"N": n_rows, "pair_set_hash": pair_set, "order_by_pair_id_asserted": True},
  "artifacts": ARTIFACTS,
  "preregistration": {"subsets": {s: ST[s]["n"] for s in SUBSETS}, "variants": list(VARIANTS),
                      "declared_before_fitting": True},
  "variant_shape_check": VARIANT_SHAPE, "span_check_V2_vs_V0_M3": SPAN_CHECK,
  "structural_pruning": {"rule": ("columns with zero variance or no observed support inside a subset "
                                  "are dropped before fitting; the same deterministic rule is applied "
                                  "to every subset, and no coefficient, p-value, R2 or performance "
                                  "figure participates in the decision"),
                         "tolerance": ZV_TOL, "zero_variance_columns_removed": ZV_REMOVED,
                         "per_design": PRUNE_LOG,
                         "not_identifiable_after_pruning": sorted(NOT_IDENT),
                         "not_identifiable_count": len(NOT_IDENT),
                         "subset_rank_fail_primary_variant_V0": SUBSET_FAIL_V0,
                         "subset_touched_by_any_variant": SUBSET_FAIL_ANY_VARIANT,
                         "cause_note": ("Two distinct causes. SOURCE_026 does not contain the frozen "
                            "reference cell 025-...-other, so the remaining cell dummies sum to one "
                            "and saturate the intercept — a coding consequence of the frozen "
                            "reference, not an empty column, and it is NOT repaired here by "
                            "re-referencing because that would change the frozen coding rule. "
                            "V3_M3DENS_LOGMCB/M3 fails on the exact byte identity and is stopped "
                            "rather than repaired by deleting a substantive variable.")},
  "bootstrap_status": BOOTSTRAP_STATUS,
  "rank_check": RANKCHECK, "blocked_variants": BLOCKED,
  "sensitivity_rows": len(ROWS), "claims": CLAIMS, "rq1_by_subset": RQ1,
  "huber": HUBER, "quantile": QUANTILE_STATUS,
  "ssot_29_reporting_fields": SSOT29, "claim_restrictions": RESTRICTIONS,
  "runtime_telemetry": {"materialize": TEL_MAT, "sufficient_statistics": TEL_SS, "huber_irls": TEL_HUB},
  "software": {"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__,
               "duckdb": duckdb.__version__, "pyarrow": pyarrow.__version__}}
MANIFEST = {"artifact_id": "NB11_RUN_MANIFEST_v001", **SHAS, "code_sha": CODE_SHA,
  "notebook": "notebooks/11_robustness.ipynb", "N": n_rows, "pair_set_hash": pair_set,
  "artifact_sha256": {k: v["sha256"] for k, v in ARTIFACTS.items()},
  "artifact_identity": f"{sum(a['match'] for a in ARTIFACTS.values())}/5",
  "subset_n": {s: ST[s]["n"] for s in SUBSETS}, "variants": list(VARIANTS),
  "blocked_variants": BLOCKED, "dependence": CLAIMS["dependence"], "bootstrap_status": BOOTSTRAP_STATUS,
  "zero_variance_columns_removed": ZV_REMOVED,
  "subset_rank_fail_after_structural_pruning_primary_variant_V0": SUBSET_FAIL_V0,
  "subset_touched_by_any_variant_non_identifiability": SUBSET_FAIL_ANY_VARIANT,
  "designs_not_identifiable_after_pruning": len(NOT_IDENT),
  "huber_iterations": HUBER_ITERS, "huber_tuning_constant": HUBER_K,
  "quantile": QUANTILE_STATUS["status"],
  "runtime_sec": {k: v.get("elapsed_sec") for k, v in
                  (("materialize", TEL_MAT), ("suffstat", TEL_SS), ("huber", TEL_HUB))},
  "eng_obs_001": {k: {kk: v.get(kk) for kk in ("interval_sec", "sample_count", "r1_periodic_sampling",
                       "min_mem_available_gib", "peak_rss_gib", "worst_memory_status",
                       "red_or_worse_sample_count", "final_status")}
                  for k, v in (("materialize", TEL_MAT), ("suffstat", TEL_SS), ("huber", TEL_HUB))},
  "ssot_29_reporting_fields": SSOT29, "claim_restrictions": RESTRICTIONS, "raw_text_persisted": False}

def dump(rel, obj):
    (ROOT / rel).write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False) + "\n", "utf-8"); print("wrote", rel)
dump("outputs/reports/NB11_ROBUSTNESS_SUMMARY_v001.json", SUMMARY)
dump("outputs/manifests/NB11_RUN_MANIFEST_v001.json", MANIFEST)
keys = sorted({k for r in ROWS for k in r})
with open(ROOT / "outputs/tables/NB11_SENSITIVITY_MATRIX_v001.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(ROWS)
print("wrote outputs/tables/NB11_SENSITIVITY_MATRIX_v001.csv")

L = ["# NB11 — Robustness / Sensitivity", "",
     "NB11 tests the stability of the NB09 primary results. It does not redefine them.",
     "Conditional associations only; no causal claim.", "",
     "```", f"N = {n_rows:,}   pair-set {pair_set}",
     "subsets  " + ", ".join(f"{s}={ST[s]['n']:,}" for s in SUBSETS),
     f"variants {', '.join(VARIANTS)}", "```", "", "## Claim register", "",
     "| claim | direction | magnitude | effect range |", "|---|---|---|---|"]
for k in ["RQ1", "RQ3", "RQ4", "RQ5"]:
    c = CLAIMS[k]
    if k == "RQ1":
        er = c["effect_range"]; rngtxt = (f"median TP {er['median_TP_min']:.4f}–{er['median_TP_max']:.4f}, "
                                          f"P(TP>1) {er['share_TP_gt_1_min']:.4f}–{er['share_TP_gt_1_max']:.4f}")
    else:
        a = c["outcome_A_range"]; rngtxt = (f"A ΔR² {a['delta_r2_min']:.6f}–{a['delta_r2_max']:.6f}, "
                                            f"partial {a['partial_r2_min']:.6f}–{a['partial_r2_max']:.6f} "
                                            f"({a['n_specifications']} specs)")
    L.append(f"| {k} — {c['claim']} | {c['direction']} | {c['magnitude']} | {rngtxt} |")
L += ["", "## Structural pruning and non-identifiability", "",
      f"- zero-variance / unsupported columns removed before fitting: **{ZV_REMOVED}**",
      f"- designs still not identifiable after pruning: **{len(NOT_IDENT)}** of {len(PRUNE_LOG)}",
      f"- subsets where the primary specification V0 is not identifiable: "
      f"**{', '.join(SUBSET_FAIL_V0) or 'none'}**",
      "- `SOURCE_026` lacks the frozen reference cell, so the remaining cell dummies saturate the "
      "intercept. That is a consequence of the frozen coding, not an empty column, and is not "
      "repaired here by re-referencing.",
      "- `V3_M3DENS_LOGMCB` at M3 fails on the exact byte identity and is stopped, not repaired by "
      "deleting a substantive variable.", "",
      "## Not testable / bounded", ""]
for k in ["SSOT_24_1_raw_text", "SSOT_24_2_nfc_only", "SSOT_24_3_curated_source",
          "SSOT_24_9_source_random", "dependence"]:
    L.append(f"- **{k}** — `{CLAIMS[k]['status']}`: {CLAIMS[k].get('reason', CLAIMS[k].get('evidence'))}")
L += ["", "## Binding restrictions", "", "```"] + [f"{k} = {v}" for k, v in RESTRICTIONS.items()] + ["```", ""]
(ROOT / "docs/results/nb11/README.md").write_text("\n".join(L) + "\n", "utf-8")
print("wrote docs/results/nb11/README.md")
print("NB11_ROBUSTNESS_EXECUTION_COMPLETE")

wrote outputs/reports/NB11_ROBUSTNESS_SUMMARY_v001.json
wrote outputs/manifests/NB11_RUN_MANIFEST_v001.json
wrote outputs/tables/NB11_SENSITIVITY_MATRIX_v001.csv
wrote docs/results/nb11/README.md
NB11_ROBUSTNESS_EXECUTION_COMPLETE
